# Track 4 — Smart City (Transport / Public Data)
### EP0418 · AI for Real-World Data and Automated Decisions · Starter Pack

**Who this suits:** SOC students, urban-curious, Singapore-grounded projects.
**Data:** 8 weeks of hourly taxi availability across Singapore, pulled live from
`data.gov.sg` — **no API key required**.
**Macro lesson:** *Strong seasonality and the value of decomposition.*
**Your built-in baseline:** seasonal-naive — "same hour, same day, last week".

---
This is the cleanest seasonality in any of the six tracks. You will see the daily
cycle immediately and the weekly cycle right behind it. That makes the
seasonal-naive baseline genuinely hard to beat — which is exactly the point
(SLO 6.2).

> **Want LTA DataMall instead?** Bus arrivals, traffic speed bands and carpark
> availability need a free Account Key from
> <https://datamall.lta.gov.sg> — register in **Week 1**, approval is not instant.
> DataMall is real-time only, so you must collect your own history.

> **First: File → Save a copy in Drive, and work in your copy.** This page opened from GitHub, and changes here are not saved. Reopen your copy from Google Drive next time.


In [ ]:
import os, urllib.request
# Fetch this pack's data from the course repository if it is not here yet.
for _f in ['sg_taxi_hourly.csv']:
    if not os.path.exists(_f):
        urllib.request.urlretrieve("https://raw.githubusercontent.com/StanleySP-eng/EP0418-labs/main/starter_packs/track4_smart_city/" + _f, _f)

import pandas as pd, numpy as np
import matplotlib.pyplot as plt
import os

CSV = "sg_taxi_hourly.csv"
if not os.path.exists(CSV):
    from google.colab import files; files.upload()      # pick the CSV from the pack

df = pd.read_csv(CSV, parse_dates=["timestamp"]).sort_values("timestamp").reset_index(drop=True)
df["hour"] = df.timestamp.dt.hour
df["dow"]  = df.timestamp.dt.dayofweek

In [ ]:
# --- FIRST LOOK -------------------------------------------------------
print("rows:", len(df), " range:", df.timestamp.min(), "->", df.timestamp.max())
print("\ntaxi_count:"); print(df.taxi_count.describe().round(0).to_string())
print("\nfirst five raw timestamps:")
print(df.timestamp.head().to_string(index=False))

### Look at those timestamps before you go further

They are **not** on the hour: `23:59:48`, `00:58:55`, `01:59:31`. The API returns
whenever the reading was actually taken, not a tidy clock tick. This is normal for
real feeds and it is exactly the *irregular sampling interval* problem in SLO 3.2.

It matters immediately: if you try `df.shift(168)` to get "the same hour last
week", you are shifting by 168 **rows**, and rows are not evenly spaced in time.
Resample to a regular grid first (SLO 3.3) — that is the next cell.

Also check the minimum in `describe()`. A reading of ~24 available taxis
island-wide is not plausible; it is a feed glitch. Decide what to do with it.

In [ ]:
# --- RESAMPLE TO A REGULAR HOURLY GRID (SLO 3.3) ----------------------
s = df.set_index("timestamp").taxi_count.resample("h").mean()
print(f"raw rows {len(df)}  ->  {len(s)} evenly-spaced hourly slots")
print(f"empty slots created by resampling: {s.isna().sum()}")

In [ ]:
# --- PLOT: the series, and the two cycles inside it -------------------
fig = plt.figure(figsize=(13, 8))
a1 = plt.subplot2grid((3, 2), (0, 0), colspan=2)
a1.plot(df.timestamp, df.taxi_count, lw=.7)
a1.set_title("Available taxis — 8 weeks, hourly"); a1.grid(alpha=.3)

a2 = plt.subplot2grid((3, 2), (1, 0), rowspan=2)
df.groupby("hour").taxi_count.mean().plot(ax=a2, marker="o")
a2.set_title("Daily cycle (mean by hour)"); a2.set_xlabel("hour"); a2.grid(alpha=.3)

a3 = plt.subplot2grid((3, 2), (1, 1), rowspan=2)
df.groupby("dow").taxi_count.mean().plot(ax=a3, marker="o", color="tab:green")
a3.set_title("Weekly cycle (mean by weekday)")
a3.set_xticks(range(7)); a3.set_xticklabels(["Mon","Tue","Wed","Thu","Fri","Sat","Sun"])
a3.grid(alpha=.3)
plt.tight_layout(); plt.show()

In [ ]:
# --- YOUR BASELINE: SEASONAL-NAIVE ------------------------------------
# Two one-line baselines, scored across forecast horizons. No training.
#   persistence    = "it will be whatever it was h hours ago"
#   seasonal-naive = "it will be whatever it was at this hour LAST WEEK"
# For a horizon of h hours you may only use data from h hours ago — so
# persistence has to shift by h. Seasonal-naive always shifts by 168.

print(f"mean taxi count: {s.mean():.0f}\n")
print(f"{'horizon':>8}  {'persistence':>12}  {'seasonal-naive':>15}   winner")
for h in [1, 3, 6, 12, 24, 48, 72]:
    p, sn = s.shift(h), s.shift(168)
    m = s.notna() & p.notna() & sn.notna()
    a = (s[m] - p[m]).abs().mean()
    b = (s[m] - sn[m]).abs().mean()
    print(f"{h:6d}h  {a:12.1f}  {b:15.1f}   {'persistence' if a < b else 'SEASONAL-NAIVE'}")

print("\n>>> Read that table before you build anything.")
print(">>> Persistence wins at 1 hour, then falls apart as the horizon grows.")
print(">>> Seasonal-naive barely moves — it does not care how far ahead you ask.")
print(">>> So there is no single 'the baseline'. There is a baseline PER HORIZON,")
print(">>> and you must beat the best one at the horizon YOUR system actually uses.")

---
## Three macro questions to investigate

**1. What horizon does your system actually need — and which baseline does that make?**
The table above is the most important output in this notebook. If your alert
fires an hour ahead, you must beat *persistence*, and that is hard. If it plans a
day ahead, you must beat *seasonal-naive*. Students who quote a baseline at the
wrong horizon can claim a win they did not earn. State your horizon first, then
pick your bar (SLO 6.1, 6.2).

**2. What does a residual plot tell you that a score does not?**
Subtract the seasonal-naive prediction from the actual and plot the leftovers.
Flat noise means the cycle is fully captured. Structure in the residuals — a
specific weekday, a spike at a specific hour — is your model's opportunity, and
it points straight at the feature you are missing (SLO 3.1).

**3. What decision does a taxi forecast actually drive?**
Forecasting supply is only useful if something changes as a result (SLO 7.2).
Who acts on it — a driver deciding where to go, a commuter deciding whether to
book, an operator repositioning a fleet? Pick one, then define the threshold that
triggers your alert and justify why it is set where it is.

## Now hand over to your AI assistant

Everything above was written for you. From here you drive. Open ChatGPT, Claude,
Copilot, or Cursor and work the **prompt → generate → verify → refine** loop
(SLO 9.1). A prompt worth copying looks like this:

> I have a pandas DataFrame `df` with columns `timestamp, taxi_count, hour, dow`.
> The time column is `timestamp`. Decompose this hourly series into trend, daily seasonality, weekly seasonality and residual using statsmodels STL, then plot each component separately.
> Show me the code and explain what each step does.

**Then verify before you trust it (SLO 9.2).** Every time:

| Check | How |
|---|---|
| Does it run? | Run the cell. Read the error, don't just re-prompt. |
| Right shape? | `df.shape` before and after — did you lose rows you didn't mean to? |
| Right range? | `df.describe()` — any impossible values (negative rainfall, 300 bpm)? |
| Spot-check | Pick 2 rows by hand and confirm the number is what you'd compute yourself. |
| Beat baseline? | Compare against the baseline above. If it loses, say so honestly. |

**Keep a `verification_log.md`** as you go — one line per AI mistake you caught and
how you fixed it. That log is assessed in CA3, and an empty one reads as
"I did not check", not "the AI was perfect".